In [1]:
import os, json, re
import asyncio
from typing import List, Dict, Any, Optional, Literal
from datetime import datetime
from IPython.display import Markdown, display

from pydantic import BaseModel, Field, ValidationError

from agents import Agent, Runner, Trace
from agents.mcp import MCPServerStdio

from dotenv import load_dotenv

load_dotenv()

True

In [2]:

# =========================
# MCP SERVERS + RUNTIME SERVICES
# =========================

# MSSQL MCP
sql_params = {
    "command": "mssql-mcp-server",
    "env": {"MSSQL_CONNECTION_STRING": os.getenv("MSSQL_CONNECTION_STRING")},
}

# Chart MCP (optional, used only when a chart is clearly helpful)
chart_params = {"command": "npx", "args": ["-y", "@antv/mcp-server-chart"]}

mssql_server = MCPServerStdio(params=sql_params, client_session_timeout_seconds=60)
chart_server = MCPServerStdio(params=chart_params, client_session_timeout_seconds=60)

# Deterministic local memory store (Python-managed, not agent-managed)
CHROMA_DIR = os.getenv("CHROMA_DIR", "./chroma_ai_analytics")


In [3]:

# =========================
# PYDANTIC OUTPUTS
# =========================

class PlannerDecision(BaseModel):
    require_schema_refresh: bool = True
    include_chart: bool = False
    rationale: str = ""


class CandidateTable(BaseModel):
    name: str
    confidence: float = Field(ge=0, le=1)
    why: str


class ColumnInfo(BaseModel):
    col: str
    type: Optional[str] = None


class TableColumnsEntry(BaseModel):
    table: str
    columns: List[ColumnInfo] = Field(default_factory=list)


class EavHintLite(BaseModel):
    table: str
    attribute_cols: List[str] = Field(default_factory=list)
    value_cols: List[str] = Field(default_factory=list)


class DBCatalogLite(BaseModel):
    candidate_fact_tables: List[CandidateTable] = Field(default_factory=list)
    table_columns: List[TableColumnsEntry] = Field(default_factory=list)
    eav_hints: List[EavHintLite] = Field(default_factory=list)
    notes: List[str] = Field(default_factory=list)


JsonCell = str | int | float | bool | None


class AnswerTable(BaseModel):
    name: str
    columns: List[str]
    rows: List[List[JsonCell]]


class SQLQuery(BaseModel):
    name: str
    sql: str


class QueryRunResult(BaseModel):
    answer_tables: List[AnswerTable]
    assumptions: List[str] = Field(default_factory=list)
    caveats: List[str] = Field(default_factory=list)
    sql_appendix: List[SQLQuery]


class VerificationResult(BaseModel):
    verdict: Literal["PASS", "FAIL"]
    reasons: List[str] = Field(default_factory=list)
    fixes: List[str] = Field(default_factory=list)


class NotebookRunResult(BaseModel):
    final_output: str
    verifier_verdict: Literal["PASS", "FAIL"]
    verification_reasons: List[str] = Field(default_factory=list)


In [4]:

# =========================
# HELPERS: JSON, MEMORY, SQL SAFETY
# =========================

import chromadb
from chromadb.utils import embedding_functions


def collection_name_for(db_key: str) -> str:
    safe = re.sub(r"[^a-zA-Z0-9_\-]", "_", db_key)[:60]
    return f"ai_analytics__{safe}"


def normalize_model_output(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.I)
        text = re.sub(r"\s*```$", "", text.strip())
    return text.strip()


def parse_model(model_cls, text: str, label: str = "model_output"):
    normalized = normalize_model_output(text)
    if not normalized:
        raise ValueError(f"No output received for {label}")
    return model_cls.model_validate_json(normalized)


def _strip_sql_comments(sql: str) -> str:
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    sql = re.sub(r"--.*?$", " ", sql, flags=re.M)
    return sql


def check_select_only_sql(sql: str) -> Optional[str]:
    s = _strip_sql_comments(sql).strip()
    if not s:
        return "Empty SQL is not allowed"

    if ";" in s[:-1]:
        return "Only single-statement SQL is allowed"

    head = s.lower().lstrip("(")
    if not (head.startswith("select") or head.startswith("with")):
        return "SQL must start with SELECT or WITH"

    banned = re.compile(
        r"\b(insert|update|delete|merge|drop|alter|truncate|create|exec|execute|xp_cmdshell|sp_configure)\b",
        flags=re.I,
    )
    if banned.search(s):
        return "Unsafe SQL keyword detected"

    if re.search(r"\bselect\b[\s\S]*\binto\b", s, flags=re.I):
        return "SELECT INTO is not allowed"

    return None


def enforce_sql_safety(qres: QueryRunResult):
    for q in qres.sql_appendix:
        reason = check_select_only_sql(q.sql)
        if reason:
            raise ValueError(f"Unsafe SQL in '{q.name}': {reason}")


chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
_openai_key = os.getenv("OPENAI_API_KEY")
_embedding_fn = None
if _openai_key:
    _embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=_openai_key,
        model_name=os.getenv("CHROMA_OPENAI_EMBED_MODEL", "text-embedding-3-small"),
    )


def get_collection(db_key: str):
    return chroma_client.get_or_create_collection(
        name=collection_name_for(db_key),
        embedding_function=_embedding_fn,
        metadata={"db_key": db_key},
    )


def rag_retrieve(db_key: str, query: str, k: int = 8, max_chars: int = 2500) -> str:
    col = get_collection(db_key)
    res = col.query(query_texts=[query], n_results=k)
    docs = (res.get("documents") or [[]])[0] or []
    return ("\n\n".join(docs[:k]))[:max_chars]


def rag_upsert_docs(db_key: str, items: List[Dict[str, Any]]):
    if not items:
        return
    col = get_collection(db_key)
    col.upsert(
        ids=[x["id"] for x in items],
        documents=[x["doc"] for x in items],
        metadatas=[x.get("meta", {}) for x in items],
    )




def rag_upsert_catalog(db_key: str, catalog: DBCatalogLite):
    items = []
    now = datetime.utcnow().isoformat()

    for entry in catalog.table_columns:
        t = entry.table
        cols = entry.columns
        items.append({
            "id": f"table::{t}",
            "doc": (
                f"TABLE {t}\n"
                f"Columns: {', '.join(c.col for c in cols[:120])}"
            ),
            "meta": {"type": "table", "table": t, "db_key": db_key, "updated_at": now},
        })

    for e in catalog.eav_hints:
        items.append({
            "id": f"eav::{e.table}",
            "doc": (
                f"EAV_HINT {e.table}\n"
                f"attribute_cols={e.attribute_cols}\n"
                f"value_cols={e.value_cols}"
            ),
            "meta": {"type": "eav_hint", "table": e.table, "db_key": db_key, "updated_at": now},
        })

    rag_upsert_docs(db_key, items)



def rag_upsert_semantics(db_key: str, user_request: str, qres: QueryRunResult, catalog: Optional[DBCatalogLite] = None):
    now = datetime.utcnow().isoformat()
    items = []

    for q in qres.sql_appendix:
        items.append({
            "id": f"sql::{q.name}::{abs(hash(q.sql)) % (10**10)}",
            "doc": f"REQUEST: {user_request}\nQUERY_NAME: {q.name}\nSQL: {q.sql}",
            "meta": {"type": "sql_pattern", "db_key": db_key, "updated_at": now},
        })

    if catalog:
        for ct in catalog.candidate_fact_tables[:5]:
            items.append({
                "id": f"fact::{ct.name}",
                "doc": f"FACT_CANDIDATE {ct.name}; confidence={ct.confidence}; why={ct.why}",
                "meta": {"type": "fact_candidate", "table": ct.name, "db_key": db_key, "updated_at": now},
            })

    rag_upsert_docs(db_key, items)


In [5]:
PLANNER_INSTRUCTIONS = """
You are Planner for an AI analytics workflow.

Goal:
- decide if schema discovery is needed
- decide if a chart is useful

Input:
- semantic_db_key
- user_request
- rag_context (optional)

Output:
Return JSON only:
{
  "require_schema_refresh": true/false,
  "include_chart": true/false,
  "rationale": "short reason"
}

Rules:
- require_schema_refresh=true when:
  - rag_context is empty/weak, OR
  - user request references unfamiliar tables/fields, OR
  - EAV-like interpretation is likely.
- include_chart=true only for clear trend/comparison/ranking outputs.
- Keep rationale short and concrete.
"""

SQL_ANALYST_INSTRUCTIONS = """
You are an expert SQL Analyst. You handle either DISCOVER or ANALYZE mode.

Allowed tools:
- execute_query
- list_tables
- describe_table
- sample_data

Input JSON:
- task_mode: DISCOVER | ANALYZE
- semantic_db_key
- user_request
- rag_context (optional)
- catalog (optional, ANALYZE mode)
- verifier_fixes (optional, ANALYZE mode)

Global rules:
- Work autonomously; do not ask user for permission.
- Do not claim data is unavailable until discovery is complete and at least one relevant analysis query has been attempted.
- Prefer aggregate outputs; avoid obvious PII columns.
- Use explicit ORDER BY for trends/rankings.
- For “latest/most recent”, compute explicitly via MAX(date_col).
- Discovery must be lightweight and targeted.
- Do NOT run full relationship graph scans.
- Do NOT use SELECT * in discovery queries.
- Only inspect tables relevant to the user request and likely neighbors.

DISCOVER mode (strict):
1) Query INFORMATION_SCHEMA.TABLES and INFORMATION_SCHEMA.COLUMNS first.
2) Identify likely fact-like tables and key candidate tables relevant to the request (targeted subset only).
3) Detect EAV patterns minimally:
   - attribute key/name columns
   - value columns
4) Use describe_table first; if semantic confirmation is required, run exactly one minimal `SELECT TOP 1 ...` (or equivalent single-row sample) per ambiguous table.
5) Return JSON only matching DBCatalogLite.

DBCatalogLite shape reminders:
- candidate_fact_tables
- table_columns is a LIST of objects: {table, columns:[{col,type}]}
- eav_hints is a LIST of objects: {table, attribute_cols:[...], value_cols:[...]}
- notes
- Do not return dict maps keyed by table name.

DISCOVER output caps (mandatory):
- candidate_fact_tables: at most 6
- table_columns: at most 8 tables
- columns per table_columns entry: at most 20 columns
- eav_hints: at most 3
- attribute_cols per eav_hints entry: at most 8
- value_cols per eav_hints entry: at most 8
- notes: at most 8

DISCOVER process caps (mandatory):
- Total tool calls in DISCOVER: at most 6
- INFORMATION_SCHEMA queries: exactly 2 (TABLES + COLUMNS)
- describe_table calls: at most 4
- SELECT TOP 1 sample queries: at most 2 total
- If ambiguity remains after these caps, write uncertainty in notes and return immediately.

DISCOVER execution sequence (mandatory):
1) Run INFORMATION_SCHEMA.TABLES once.
2) Run INFORMATION_SCHEMA.COLUMNS once.
3) Select targeted candidate tables only.
4) Run up to 4 describe_table calls.
5) Run up to 2 SELECT TOP 1 checks only if still ambiguous.
6) Return DBCatalogLite immediately; do not continue exploration.

6) Stop discovery as soon as DBCatalogLite can be populated for the current request.
   - If uncertain, record uncertainty in notes and return.

ANALYZE mode (strict):
1) Build a query plan from request (metric, grain, window, segments).
2) Use catalog/rag_context hints if present.
3) Complex SQL is allowed: CTE, window functions, conditional aggregates, subqueries, pivot for EAV.
4) Keep result tables compact and chart-friendly.
5) Return JSON only matching QueryRunResult with exact SQL in sql_appendix.
"""

VERIFIER_INSTRUCTIONS = """
You are Verifier. No tools.

Input:
- user_request
- query_run_result (answer_tables, assumptions, caveats, sql_appendix)

Checks:
1) Safety:
- SELECT/WITH only.
- No INSERT/UPDATE/DELETE/MERGE/CREATE/ALTER/DROP/TRUNCATE.
- No EXEC/xp_cmdshell/sp_configure/openrowset/opendatasource.
- No multi-statement SQL.
2) Correctness:
- Time series ordered by date.
- Rankings ordered by metric.
- Date window aligns to request.
- Flag likely join duplication risks.
3) Output quality:
- aggregate-first output
- assumptions/caveats explicit if ambiguity exists

Return JSON only matching VerificationResult.
"""

NARRATOR_INSTRUCTIONS = """
You are Narrator. No DB queries.

Input:
- answer_tables
- assumptions
- caveats
- sql_appendix
- optional chart_artifact.image_url

Output:
- Executive Summary
- Key Insights
- Data Quality Notes
- Technical Notes (short)
- SQL Appendix (exact SQL)

If chart_artifact.image_url exists, include:
## Chart
![Chart](<image_url>)
"""


In [6]:

# # =========================
# # AGENT INSTRUCTIONS (LEAN + TESTABLE)
# # =========================

# PLANNER_INSTRUCTIONS = """
# You are a planning-only orchestrator for analytics over unknown databases.
# Return JSON only with this shape:
# {
#   "require_schema_refresh": true/false,
#   "include_chart": true/false,
#   "rationale": "short reason"
# }
# Rules:
# - require_schema_refresh=true when RAG context is empty/weak or request looks schema-sensitive (new domain terms, EAV hints, unfamiliar tables).
# - include_chart=true only if output likely benefits from one chart (time trend/ranking/comparison).
# - Keep rationale under 30 words.
# """

# SQL_ANALYST_INSTRUCTIONS = """
# You are SQL Analyst. You perform both schema discovery and analysis.

# You receive JSON input with:
# - task_mode: DISCOVER or ANALYZE
# - semantic_db_key (memory key only, not DB connection name)
# - user_request
# - rag_context (optional)
# - catalog (optional)
# - verifier_fixes (optional)

# Allowed tools: execute_query, list_tables, describe_table, sample_data, get_relationships.

# Connection rule:
# - semantic_db_key is for memory/routing context only.
# - Do NOT assume semantic_db_key is a DB named connection.
# - Call DB tools using default connection behavior unless explicit connection_name is provided in input.

# If task_mode == DISCOVER:
# - Discover schema primarily via INFORMATION_SCHEMA (tables + columns).
# - Identify candidate fact/date/measure columns, likely joins, and EAV hints.
# - Use describe_table and minimal sample_data only when needed.
# - Return JSON only matching DBCatalog.

# If task_mode == ANALYZE:
# - Answer user_request using SQL and execute_query.
# - Use catalog/rag_context as hints.
# - Complex SQL is allowed (CTE/window/conditional aggregates/pivot for EAV).
# - Always ORDER BY for trends/rankings.
# - For "latest/most recent", compute explicitly with MAX(date_col).
# - Prefer aggregates; avoid row-level PII outputs.
# - Return JSON only matching QueryRunResult.
# - sql_appendix must include exact executed SQL.
# """

# VERIFIER_INSTRUCTIONS = """
# You are Verifier. No tools.
# Input contains user_request + QueryRunResult.
# Return JSON only matching VerificationResult.
# Checks:
# 1) Safety: SELECT/WITH only, no DDL/DML/EXEC patterns.
# 2) Correctness: proper ordering, time window alignment, latest-date logic when requested.
# 3) Data handling: aggregate-first, assumptions/caveats explicit when ambiguous.
# """

# NARRATOR_INSTRUCTIONS = """
# You are Narrator. No tools.
# Input JSON includes answer_tables, assumptions, caveats, sql_appendix, and optional chart_artifact.image_url.
# Write concise business prose with sections:
# - Executive Summary
# - Key Insights
# - Data Quality Notes
# - Technical Notes
# - SQL Appendix
# If chart_artifact.image_url exists, include:
# ## Chart
# ![Chart](<image_url>)
# """


In [7]:

# =========================
# BUILD AGENTS (SIMPLIFIED)
# =========================


def build_agents(db_key: str):
    planner = Agent(
        name="Planner",
        model="gpt-4o-mini",
        instructions=PLANNER_INSTRUCTIONS,
        output_type=PlannerDecision,
        mcp_servers=[],
    )

    sql_analyst_discover = Agent(
        name="SQL Analyst Discover",
        model="gpt-5.2",#"gpt-4o-mini",
        instructions=SQL_ANALYST_INSTRUCTIONS,
        output_type=DBCatalogLite,
        mcp_servers=[mssql_server],
    )

    sql_analyst_query = Agent(
        name="SQL Analyst Query",
        model="gpt-5.2",#"gpt-4o-mini",
        instructions=SQL_ANALYST_INSTRUCTIONS,
        output_type=QueryRunResult,
        mcp_servers=[mssql_server],
    )

    verifier = Agent(
        name="Verifier",
        model="gpt-4o-mini",
        instructions=VERIFIER_INSTRUCTIONS,
        output_type=VerificationResult,
        mcp_servers=[],
    )

    narrator = Agent(
        name="Narrator",
        model="gpt-4o-mini",
        instructions=NARRATOR_INSTRUCTIONS,
        mcp_servers=[],
    )

    return {
        "planner": planner,
        "sql_analyst_discover": sql_analyst_discover,
        "sql_analyst_query": sql_analyst_query,
        "verifier": verifier,
        "narrator": narrator,
    }


In [8]:

# =========================
# END-TO-END PIPELINE
# =========================


async def run_structured(
    agent: Agent,
    payload: Dict[str, Any],
    model_cls,
    *,
    debug_label: str,
    max_turns: int,
):
    res = await Runner.run(agent, json.dumps(payload), max_turns=max_turns)
    out = res.final_output

    if isinstance(out, model_cls):
        return out
    if isinstance(out, dict):
        return model_cls.model_validate(out)
    if isinstance(out, str):
        return parse_model(model_cls, out, label=debug_label)
    raise ValueError(f"{debug_label}: unexpected output type {type(out)}")


async def run_planner(agents: Dict[str, Agent], db_key: str, user_prompt: str, rag_ctx: str) -> PlannerDecision:
    payload = {
        "semantic_db_key": db_key,
        "user_request": user_prompt,
        "rag_context": rag_ctx,
    }
    return await run_structured(
        agents["planner"],
        payload,
        PlannerDecision,
        debug_label="Planner",
        max_turns=8,
    )


async def run_sql_analyst_discover(agents: Dict[str, Agent], db_key: str, user_prompt: str, rag_ctx: str) -> DBCatalogLite:
    payload = {
        "task_mode": "DISCOVER",
        "semantic_db_key": db_key,
        "user_request": user_prompt,
        "rag_context": rag_ctx,
    }
    catalog = await run_structured(
        agents["sql_analyst_discover"],
        payload,
        DBCatalogLite,
        debug_label="SQL Analyst DISCOVER",
        max_turns=20,
    )
    rag_upsert_catalog(db_key, catalog)
    return catalog


async def run_sql_analyst_query(
    agents: Dict[str, Agent],
    db_key: str,
    user_prompt: str,
    rag_ctx: str,
    catalog: Optional[DBCatalogLite],
    verifier_fixes: Optional[List[str]] = None,
) -> QueryRunResult:
    payload = {
        "task_mode": "ANALYZE",
        "semantic_db_key": db_key,
        "user_request": user_prompt,
        "rag_context": rag_ctx,
        "catalog": catalog.model_dump() if catalog else None,
        "verifier_fixes": verifier_fixes or [],
    }
    qres = await run_structured(
        agents["sql_analyst_query"],
        payload,
        QueryRunResult,
        debug_label="SQL Analyst ANALYZE",
        max_turns=30,
    )
    enforce_sql_safety(qres)
    return qres


async def run_verifier(agents: Dict[str, Agent], user_request: str, qres: QueryRunResult) -> VerificationResult:
    payload = {
        "semantic_db_key": "",
        "user_request": user_request,
        "query_run_result": qres.model_dump(),
    }
    return await run_structured(
        agents["verifier"],
        payload,
        VerificationResult,
        debug_label="Verifier",
        max_turns=8,
    )


async def maybe_generate_chart(qres: QueryRunResult, include_chart: bool) -> Optional[Dict[str, Any]]:
    if not include_chart or not qres.answer_tables:
        return None

    t = qres.answer_tables[0]
    if len(t.columns) < 2 or not t.rows:
        return None

    x_col, y_col = t.columns[0], t.columns[1]
    data = []
    for r in t.rows[:120]:
        if len(r) < 2:
            continue
        try:
            y = float(r[1])
        except Exception:
            continue
        data.append({x_col: str(r[0]), y_col: y})

    if len(data) < 2:
        return None

    data.append({"time": str(r[0]), "value": float(r[1])})

    chart_input = {
        "data": data,
        "xField": "time",
        "yField": "value",
        "title": f"{t.name}: {y_col} by {x_col}",
    }


    result = await chart_server.call_tool("generate_line_chart", chart_input)
    image_url = result.content[0].text if result and result.content else None
    if image_url:
        return {"image_url": image_url, "spec": {"kind": "line", "x": x_col, "y": y_col}}
    return None


async def run_narrator(agents: Dict[str, Agent], db_key: str, qres: QueryRunResult, chart_artifact: Optional[Dict[str, Any]]) -> str:
    payload = {
        "db_key": db_key,
        "answer_tables": [t.model_dump() for t in qres.answer_tables],
        "assumptions": qres.assumptions,
        "caveats": qres.caveats,
        "sql_appendix": [q.model_dump() for q in qres.sql_appendix],
        "chart_artifact": chart_artifact,
    }
    res = await Runner.run(agents["narrator"], json.dumps(payload), max_turns=10)
    return res.final_output


async def run_request(user_prompt: str, db_key: str, use_narrator: bool = True) -> NotebookRunResult:
    agents = build_agents(db_key)

    rag_ctx = rag_retrieve(db_key, user_prompt, k=8, max_chars=2200)
    plan = await run_planner(agents, db_key, user_prompt, rag_ctx)

    catalog = None
    if plan.require_schema_refresh or not rag_ctx.strip():
        print("Running Discovery")
        catalog = await run_sql_analyst_discover(agents, db_key, user_prompt, rag_ctx)
        rag_ctx = rag_retrieve(db_key, user_prompt, k=10, max_chars=2200)

    qres = await run_sql_analyst_query(agents, db_key, user_prompt, rag_ctx, catalog)
    ver = await run_verifier(agents, user_prompt, qres)

    if ver.verdict == "FAIL":
        qres = await run_sql_analyst_query(
            agents,
            db_key,
            user_prompt,
            rag_ctx,
            catalog,
            verifier_fixes=ver.fixes,
        )
        ver = await run_verifier(agents, user_prompt, qres)

    if ver.verdict == "PASS":
        rag_upsert_semantics(db_key, user_prompt, qres, catalog=catalog)

    chart_artifact = await maybe_generate_chart(qres, include_chart=plan.include_chart and ver.verdict == "PASS")

    if use_narrator:
        final_output = await run_narrator(agents, db_key, qres, chart_artifact)
    else:
        final_output = json.dumps(qres.model_dump(), indent=2)

    return NotebookRunResult(
        final_output=final_output,
        verifier_verdict=ver.verdict,
        verification_reasons=ver.reasons,
    )


In [9]:

# =========================
# SMALL TEST BLOCKS
# =========================


async def test_planner(db_key: str, prompt: str):
    agents = build_agents(db_key)
    rag_ctx = rag_retrieve(db_key, prompt)
    return await run_planner(agents, db_key, prompt, rag_ctx)


async def test_discovery(db_key: str, prompt: str):
    agents = build_agents(db_key)
    rag_ctx = rag_retrieve(db_key, prompt)
    return await run_sql_analyst_discover(agents, db_key, prompt, rag_ctx)


async def test_analysis(db_key: str, prompt: str, catalog: Optional[DBCatalogLite] = None):
    agents = build_agents(db_key)
    rag_ctx = rag_retrieve(db_key, prompt)
    return await run_sql_analyst_query(agents, db_key, prompt, rag_ctx, catalog)


async def test_verify(db_key: str, prompt: str, qres: QueryRunResult):
    agents = build_agents(db_key)
    return await run_verifier(agents, prompt, qres)


async def test_narrator(db_key: str, qres: QueryRunResult):
    agents = build_agents(db_key)
    return await run_narrator(agents, db_key, qres, chart_artifact=None)


In [10]:

# =========================
# EXAMPLE USAGE
# =========================

DB_KEY = "mssql_demo"
# USER_PROMPT = (
#     "Which OCR checklist from what sites had the most negative incidents for the first month of 2025? "
#     "Use [dbo].[Checklist], [dbo].[ChecklistStructure], [dbo].[CompletedChecklist], [dbo].[CompletedChecklist_Answer]. "
#     "I suspect EAV."
# )

USER_PROMPT = (
    "Find the latest CreationDate in dbo.CompletedChecklist, then show daily completion counts "
    "for the 30 days ending on that latest date. Generate the most appropriate chart if applicable. "
    "Only generate one chart."
)

async with mssql_server, chart_server:
    # component tests:
    # plan = await test_planner(DB_KEY, USER_PROMPT)
    # catalog = await test_discovery(DB_KEY, USER_PROMPT)
    # qres = await test_analysis(DB_KEY, USER_PROMPT, catalog=catalog)
    # ver = await test_verify(DB_KEY, USER_PROMPT, qres)

    result = await run_request(USER_PROMPT, db_key=DB_KEY, use_narrator=True)
    print("Verifier:", result.verifier_verdict, result.verification_reasons[:2])
    display(Markdown(result.final_output))


/tmp/ipykernel_19216/2818086176.py:137: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat()


Verifier: PASS []


# Executive Summary
This report presents insights from a recent analysis of completion counts over a 30-day window, ending on the latest recorded date of February 6, 2025. The findings highlight daily completion trends and aggregate insights which may inform operational strategies moving forward.

# Key Insights
- The **latest completion date** recorded is **February 6, 2025**.
- **Total completion counts** over the past 30 days show variability, with significant spikes and periods of inactivity:
  - The highest completion count in a single day was **105** on **January 13, 2025**.
  - There were **several days with zero completions**, including **January 11, 12, 18, and 25, 2025**.

# Data Quality Notes
- The analysis is based on the results from a query limited to display the first 20 days due to tool restrictions. All 30 calendar days were processed in the SQL query, but only a portion of the results is displayed.
- Future requests for more complete datasets should consider adjusting settings to remove the TOP 20 limit.

# Technical Notes
- The database utilized is Microsoft SQL Server. 
- Completion counts are derived from the `dbo.CompletedChecklist` table, focusing on **CreationDate** for determining activity.

# SQL Appendix

### Fetch Latest Creation Date
```sql
SELECT MAX(CreationDate) AS LatestCreationDate
FROM dbo.CompletedChecklist;
```

### Daily Completion Counts (30-day window, with calendar spine incl. zero days)
```sql
SELECT
  cal.[Date],
  COALESCE(cnt.CompletionCount, 0) AS CompletionCount
FROM (
    SELECT DATEADD(day, v.n, b.start_date) AS [Date]
    FROM (
        SELECT
            CAST(MAX(CreationDate) AS date) AS end_date,
            DATEADD(day, -29, CAST(MAX(CreationDate) AS date)) AS start_date
        FROM dbo.CompletedChecklist
    ) b
    CROSS JOIN (VALUES
        (0),(1),(2),(3),(4),(5),(6),(7),(8),(9),
        (10),(11),(12),(13),(14),(15),(16),(17),(18),(19),
        (20),(21),(22),(23),(24),(25),(26),(27),(28),(29)
    ) v(n)
) cal
LEFT JOIN (
    SELECT
        CAST(cc.CreationDate AS date) AS [Date],
        COUNT_BIG(*) AS CompletionCount
    FROM dbo.CompletedChecklist cc
    CROSS JOIN (
        SELECT
            CAST(MAX(CreationDate) AS date) AS end_date,
            DATEADD(day, -29, CAST(MAX(CreationDate) AS date)) AS start_date
        FROM dbo.CompletedChecklist
    ) b
    WHERE cc.CreationDate >= b.start_date
      AND cc.CreationDate < DATEADD(day, 1, b.end_date)
    GROUP BY CAST(cc.CreationDate AS date)
) cnt
  ON cnt.[Date] = cal.[Date]
ORDER BY cal.[Date] ASC;
```

In [11]:

# TODO
# 1) Add a technical narrator mode for debugging traces.
# 2) Persist lightweight run telemetry (tokens, latency, SQL count) per request.
# 3) Add row-count guardrails for very large outputs and optional CSV export.
# 4) Add explicit SQL parameterization patterns for user-supplied literals.


In [15]:
# 1) Inspect Chroma collection stats + sample records
DB_KEY = "mssql_demo"
col = get_collection(DB_KEY)

print("Collection:", col.name)
print("Count:", col.count())

peek = col.peek(limit=10)  # ids, documents, metadatas
for i, _id in enumerate(peek.get("ids", []), 1):
    doc = (peek.get("documents", [""] * i)[i-1] or "")[:1000]
    meta = (peek.get("metadatas", [{}] * i)[i-1] or {})
    print(f"\n{i}. id={_id}")
    print("meta:", meta)
    print("doc:", doc)


Collection: ai_analytics__mssql_demo
Count: 6

1. id=table::dbo.CompletedChecklist
meta: {'table': 'dbo.CompletedChecklist', 'updated_at': '2026-02-14T21:09:17.526300', 'db_key': 'mssql_demo', 'type': 'table'}
doc: TABLE dbo.CompletedChecklist
Columns: CompletedChecklistID, CreationDate, SiteID, UserID, Importfile, OCRProcessedScanID, Processed, IsChecked, ChecklistStructureID, DeletionDate, DeletedBy, Score, MaxScore, ElapsedTime, FromExcel, ChecklistWorkflowTransitionID, OCRErrors, ScheduleEventID, SaveStatus, SaveDateTime

2. id=table::dbo.CompletedChecklist_Answer
meta: {'db_key': 'mssql_demo', 'updated_at': '2026-02-14T21:09:17.526300', 'table': 'dbo.CompletedChecklist_Answer', 'type': 'table'}
doc: TABLE dbo.CompletedChecklist_Answer
Columns: CompletedChecklistAnswerID, AnswerTypeID, CompletedChecklistID, QuestionID, QuestionSetID, AnswerOptionID, AnswerValue, SectionID, CreationDate, DeletionDate, CreatedBy, DeletedBy, Score, IsIdentifier, ChecklistStructureQuestionAnswerTypeID,

In [13]:
# 2) Search memory for a prompt
q = "negative incidents OCR checklist site January 2025"
res = col.query(query_texts=[q], n_results=8)
for i, doc in enumerate((res.get("documents") or [[]])[0], 1):
    meta = ((res.get("metadatas") or [[]])[0][i-1]) if (res.get("metadatas") or [[]])[0] else {}
    print(f"\n{i}. meta={meta}")
    print(doc[:500])



1. meta={'db_key': 'mssql_demo', 'type': 'table', 'updated_at': '2026-02-14T21:09:17.526300', 'table': 'dbo.CompletedChecklist'}
TABLE dbo.CompletedChecklist
Columns: CompletedChecklistID, CreationDate, SiteID, UserID, Importfile, OCRProcessedScanID, Processed, IsChecked, ChecklistStructureID, DeletionDate, DeletedBy, Score, MaxScore, ElapsedTime, FromExcel, ChecklistWorkflowTransitionID, OCRErrors, ScheduleEventID, SaveStatus, SaveDateTime

2. meta={'type': 'table', 'table': 'dbo.CompletedChecklist_Answer', 'updated_at': '2026-02-14T21:09:17.526300', 'db_key': 'mssql_demo'}
TABLE dbo.CompletedChecklist_Answer
Columns: CompletedChecklistAnswerID, AnswerTypeID, CompletedChecklistID, QuestionID, QuestionSetID, AnswerOptionID, AnswerValue, SectionID, CreationDate, DeletionDate, CreatedBy, DeletedBy, Score, IsIdentifier, ChecklistStructureQuestionAnswerTypeID, UserAnswerOptionID, RepeatingQuestionSetIndex, RepeatingQuestionIndex, IsChecked, IsAIAnswered

3. meta={'updated_at': '2026-02-14

In [14]:
import os
from pathlib import Path

print("cwd:", os.getcwd())
print("CHROMA_DIR:", CHROMA_DIR)
print("CHROMA_DIR abs:", Path(CHROMA_DIR).resolve())

for c in chroma_client.list_collections():
    print(c.name, "count=", c.count())


cwd: /home/preshen/Projects/ai_analytics
CHROMA_DIR: ./chroma_ai_analytics
CHROMA_DIR abs: /home/preshen/Projects/ai_analytics/chroma_ai_analytics
ai_analytics__mssql_demo count= 6
